# MELD (dataset) + FLAN-T5 + Prompt Engineering

Este notebook:

1. Descarga y prepara **MELD** (diálogos de *Friends*) con una **ventana de contexto** por turno.
2. Construye un `input_text` que incluye el **contexto previo** y el **turno objetivo**.
3. Evalúa un modelo **instruction-tuned** (FLAN-T5) **sin fine-tuning**, usando *prompt engineering*:
   - few-shot balanceado,
   - definiciones de etiquetas,
   - calibración contextual,
   - ensemble de prompts,
   - y *label scoring* (puntuar cada etiqueta por *loss*).

Objetivo: mejorar **precisión / macro-F1** sin entrenar el modelo.


In [1]:
# Dependencias (Colab / entorno limpio)
# Si ya las tienes instaladas, puedes omitir esta celda.
# Nota: accelerate/bitsandbytes SOLO son necesarios si quieres cuantización 8-bit/4-bit (device_map="auto").
!pip -q install -U "transformers>=4.40.0" sentencepiece kagglehub scikit-learn tqdm

# Opcional (cuantización 8-bit/4-bit):
# !pip -q install -U accelerate bitsandbytes



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import kagglehub
import os
import pandas as pd

C:\Users\Haojie\Desktop\MacOS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
path = kagglehub.dataset_download("zaber666/meld-dataset")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\Haojie\.cache\kagglehub\datasets\zaber666\meld-dataset\versions\1


In [4]:
# Cargar CSVs de texto
train_path = os.path.join(path+'/MELD-RAW/MELD.Raw/train/', "train_sent_emo.csv")
dev_path   = os.path.join(path+'/MELD-RAW/MELD.Raw/', "dev_sent_emo.csv")
test_path  = os.path.join(path+'/MELD-RAW/MELD.Raw/', "test_sent_emo.csv")

train = pd.read_csv(train_path, encoding="utf-8")
dev   = pd.read_csv(dev_path, encoding="utf-8")
test  = pd.read_csv(test_path, encoding="utf-8")

In [5]:
# Limpieza mínima de caracteres raros (apóstrofo 0x92)
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return s
    return s.replace('\x92', "")#

for df_ in (train, dev, test):
    df_["Utterance"] = df_["Utterance"].apply(clean_text)


In [6]:
# Unificar splits y quedarnos con las columnas relevantes

##%%
df = pd.concat(
    [
        train.assign(split="train"),
        dev.assign(split="dev"),
        test.assign(split="test"),
    ],
    ignore_index=True,
)

df = df[
    [
        "Dialogue_ID",
        "Utterance_ID",
        "Speaker",
        "Utterance",
        "Emotion",
        "Sentiment",
        "split",
    ]
]

print(df.head())
print(df.shape)


   Dialogue_ID  Utterance_ID          Speaker  \
0            0             0         Chandler   
1            0             1  The Interviewer   
2            0             2         Chandler   
3            0             3  The Interviewer   
4            0             4         Chandler   

                                           Utterance   Emotion Sentiment  \
0  also I was the point person on my companys tra...   neutral   neutral   
1                    You mustve had your hands full.   neutral   neutral   
2                            That I did. That I did.   neutral   neutral   
3       So lets talk a little bit about your duties.   neutral   neutral   
4                             My duties?  All right.  surprise  positive   

   split  
0  train  
1  train  
2  train  
3  train  
4  train  
(13708, 7)


In [7]:
# Construir estructura de diálogos ordenados

##%%
dialogues = (
    df.groupby("Dialogue_ID")
      .apply(lambda x: x.sort_values("Utterance_ID").to_dict("records"))
      .to_dict()
)

# Ejemplo de diálogo
dialogues[0][:5]

C:\Users\Haojie\AppData\Local\Temp\ipykernel_43664\503415823.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sort_values("Utterance_ID").to_dict("records"))


[{'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Chandler',
  'Utterance': 'also I was the point person on my companys transition from the KL-5 to GR-6 system.',
  'Emotion': 'neutral',
  'Sentiment': 'neutral',
  'split': 'train'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Mark',
  'Utterance': 'Why do all youre coffee mugs have numbers on the bottom?',
  'Emotion': 'surprise',
  'Sentiment': 'positive',
  'split': 'test'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Phoebe',
  'Utterance': 'Oh my God, hes lost it. Hes totally lost it.',
  'Emotion': 'sadness',
  'Sentiment': 'negative',
  'split': 'dev'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 1,
  'Speaker': 'The Interviewer',
  'Utterance': 'You mustve had your hands full.',
  'Emotion': 'neutral',
  'Sentiment': 'neutral',
  'split': 'train'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 1,
  'Speaker': 'Monica',
  'Utterance': 'What?',
  'Emotion': 'surprise',
  'Sentiment': 'negative',
  'split': 'dev'}]

In [8]:
# Normalizar hablantes (SPEAKER_0, SPEAKER_1, ... por diálogo)

##%%
def normalize_speakers(dialogue):
    speakers = list({turn["Speaker"] for turn in dialogue})
    mapping = {spk: f"SPEAKER_{i}" for i, spk in enumerate(speakers)}
    for turn in dialogue:
        turn["Speaker_Norm"] = mapping[turn["Speaker"]]
    return dialogue

dialogues_norm = {
    d_id: normalize_speakers(dialog)
    for d_id, dialog in dialogues.items()
}

dialogues_norm[0][:5]


[{'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Chandler',
  'Utterance': 'also I was the point person on my companys transition from the KL-5 to GR-6 system.',
  'Emotion': 'neutral',
  'Sentiment': 'neutral',
  'split': 'train',
  'Speaker_Norm': 'SPEAKER_3'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Mark',
  'Utterance': 'Why do all youre coffee mugs have numbers on the bottom?',
  'Emotion': 'surprise',
  'Sentiment': 'positive',
  'split': 'test',
  'Speaker_Norm': 'SPEAKER_5'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Phoebe',
  'Utterance': 'Oh my God, hes lost it. Hes totally lost it.',
  'Emotion': 'sadness',
  'Sentiment': 'negative',
  'split': 'dev',
  'Speaker_Norm': 'SPEAKER_2'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 1,
  'Speaker': 'The Interviewer',
  'Utterance': 'You mustve had your hands full.',
  'Emotion': 'neutral',
  'Sentiment': 'neutral',
  'split': 'train',
  'Speaker_Norm': 'SPEAKER_1'},
 {'Dialogue_ID': 0,
  'Utterance_ID':

In [9]:
# Construir filas con ventana de contexto (por ejemplo 2 turnos anteriores)

##%%
def build_context_rows(dialogue, window=4):
    rows = []
    dialogue = sorted(dialogue, key=lambda x: x["Utterance_ID"])
    for i, turn in enumerate(dialogue):
        ctx = dialogue[max(0, i - window):i]
        ctx_text = " ".join(
            f'{t["Speaker_Norm"]}: {t["Utterance"]}' for t in ctx
        )
        rows.append(
            {
                "dialogue_id": turn["Dialogue_ID"],
                "utt_id": turn["Utterance_ID"],
                "speaker": turn["Speaker_Norm"],
                "context": ctx_text,
                "text": turn["Utterance"],
                "emotion": turn["Emotion"],
                "sentiment": turn["Sentiment"],
                "split": turn["split"],
            }
        )
    return rows

all_rows = []
for d_id, dialog in dialogues_norm.items():
    all_rows.extend(build_context_rows(dialog, window=4))

df_model = pd.DataFrame(all_rows)
df_model.head()

,dialogue_id,utt_id,speaker,context,text,emotion,sentiment,split
0,0,0,SPEAKER_3,,also I was the point person on my companys tra...,neutral,neutral,train
1,0,0,SPEAKER_5,SPEAKER_3: also I was the point person on my c...,Why do all youre coffee mugs have numbers on t...,surprise,positive,test
2,0,0,SPEAKER_2,SPEAKER_3: also I was the point person on my c...,"Oh my God, hes lost it. Hes totally lost it.",sadness,negative,dev
3,0,1,SPEAKER_1,SPEAKER_3: also I was the point person on my c...,You mustve had your hands full.,neutral,neutral,train
4,0,1,SPEAKER_4,SPEAKER_3: also I was the point person on my c...,What?,surprise,negative,dev


In [10]:
# Construir el campo final para el prompt: contexto + turno objetivo (con speaker)
# Nota: incluir el speaker del turno actual suele ayudar a modelos instruction-tuned.

##%%
df_model["context"] = df_model["context"].fillna("").astype(str)

# Si no hay contexto previo, marcamos explícitamente "None" para evitar prompts vacíos ambiguos
ctx = df_model["context"].str.strip()
ctx = ctx.mask(ctx.eq(""), "None")

df_model["input_text"] = (
    "Dialogue context (previous turns):\n"
    + ctx
    + "\n\nTarget turn to classify (emotion of THIS utterance):\n"
    + df_model["speaker"].astype(str).str.strip()
    + ": "
    + df_model["text"].astype(str).str.strip()
)

df_model[["dialogue_id", "utt_id", "speaker", "input_text", "emotion", "split"]].head(10)


,dialogue_id,utt_id,speaker,input_text,emotion,split
0,0,0,SPEAKER_3,Dialogue context (previous turns):\nNone\n\nTa...,neutral,train
1,0,0,SPEAKER_5,Dialogue context (previous turns):\nSPEAKER_3:...,surprise,test
2,0,0,SPEAKER_2,Dialogue context (previous turns):\nSPEAKER_3:...,sadness,dev
3,0,1,SPEAKER_1,Dialogue context (previous turns):\nSPEAKER_3:...,neutral,train
4,0,1,SPEAKER_4,Dialogue context (previous turns):\nSPEAKER_3:...,surprise,dev
5,0,1,SPEAKER_0,Dialogue context (previous turns):\nSPEAKER_5:...,anger,test
6,0,2,SPEAKER_0,Dialogue context (previous turns):\nSPEAKER_2:...,neutral,test
7,0,2,SPEAKER_3,Dialogue context (previous turns):\nSPEAKER_1:...,neutral,train
8,0,3,SPEAKER_1,Dialogue context (previous turns):\nSPEAKER_4:...,neutral,train
9,0,4,SPEAKER_3,Dialogue context (previous turns):\nSPEAKER_0:...,surprise,train


In [11]:
# Lista de emociones (clases) presentes en MELD

##%%
emotions = sorted(df_model["emotion"].unique())
print("Emociones:", emotions)

# Separar splits (útil para evaluar solo en test)
train_df = df_model[df_model["split"] == "train"].reset_index(drop=True)
dev_df   = df_model[df_model["split"] == "dev"].reset_index(drop=True)
test_df  = df_model[df_model["split"] == "test"].reset_index(drop=True)

print(train_df["emotion"].value_counts())

Emociones: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
emotion
neutral     4710
joy         1743
surprise    1205
anger       1109
sadness      683
disgust      271
fear         268
Name: count, dtype: int64


# Prompt engineering (mejorado)

En esta sección **no hacemos fine-tuning**. En su lugar, mejoramos el rendimiento con técnicas de *prompt engineering*:

1. **Modelo más grande** (por defecto `flan-t5-large`), con **carga en 8-bit/4-bit** si está disponible.
2. **Few-shot balanceado**, eligiendo ejemplos **cortos** por clase (reduce truncation y hace el prompt más “limpio”).
3. **Definiciones de etiquetas** (guía semántica para separar emociones parecidas).
4. **Contextual calibration** (reduce el sesgo a etiquetas frecuentes como `neutral`).
5. **Prompt ensembling** (varias plantillas de prompt y promedio de puntuaciones).

La predicción se hace con **label scoring** (se puntúa cada etiqueta candidata por *loss*; menor = mejor).


In [12]:
import os
import random
import torch
import numpy as np
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -----------------------------
# 0) Setup
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Para GPU moderna, TF32 suele acelerar sin afectar demasiado
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

# -----------------------------
# 1) Modelo (más grande + cuantización si es posible)
# -----------------------------
# Con 16GB suele ir bien:
# - flan-t5-large en fp16
# - flan-t5-xl en 8-bit (si tienes bitsandbytes)
MODEL_CANDIDATES = [
    "google/flan-t5-xl",
    "google/flan-t5-large",
    "google/flan-t5-base",
]

# -----------------------------
# Cuantización (opcional)
# -----------------------------
# Para cuantización 8-bit/4-bit, Transformers suele requerir *accelerate* y *bitsandbytes*
# (porque usa device_map="auto"). Si no están disponibles, usamos fp16 normal en GPU.
HAS_ACCELERATE = False
try:
    import accelerate  # noqa: F401
    HAS_ACCELERATE = True
except Exception:
    HAS_ACCELERATE = False

HAS_BNB = False
try:
    import bitsandbytes as bnb  # noqa: F401
    HAS_BNB = True
except Exception:
    HAS_BNB = False

# Por defecto, intenta cuantización solo si el entorno lo soporta
PREFER_8BIT = HAS_ACCELERATE and HAS_BNB   # 8-bit (requiere accelerate+bitsandbytes)
PREFER_4BIT = False                        # 4-bit aún más ligero; actívalo si lo necesitas

if device.type == "cuda" and not (HAS_ACCELERATE and HAS_BNB):
    print("ℹ️ accelerate/bitsandbytes no detectados: se usará fp16 en GPU (sin device_map).")

def load_seq2seq_model(candidates=MODEL_CANDIDATES):
    last_err = None
    for name in candidates:
        try:
            tok = AutoTokenizer.from_pretrained(name)
            kwargs = {}

            # Intento de cuantización (solo si CUDA)
            qconf = None
            if device.type == "cuda" and (PREFER_8BIT or PREFER_4BIT) and HAS_ACCELERATE and HAS_BNB:
                try:
                    from transformers import BitsAndBytesConfig
                    if PREFER_4BIT:
                        qconf = BitsAndBytesConfig(
                            load_in_4bit=True,
                            bnb_4bit_use_double_quant=True,
                            bnb_4bit_quant_type="nf4",
                            bnb_4bit_compute_dtype=torch.float16,
                        )
                    elif PREFER_8BIT:
                        qconf = BitsAndBytesConfig(load_in_8bit=True)
                except Exception:
                    qconf = None

            if qconf is not None:
                # Requiere accelerate + bitsandbytes
                kwargs["device_map"] = "auto"
                kwargs["quantization_config"] = qconf
            else:
                # fp16 en GPU
                if device.type == "cuda":
                    kwargs["torch_dtype"] = torch.float16

            model = AutoModelForSeq2SeqLM.from_pretrained(name, **kwargs)
            if qconf is None:
                model = model.to(device)

            model.eval()
            print(f"✅ Cargado modelo: {name} (quantized={qconf is not None})")
            return tok, model, name
        except Exception as e:
            last_err = e
            print(f"⚠️ No se pudo cargar {name}: {type(e).__name__}: {e}")
            continue
    raise RuntimeError(f"No se pudo cargar ningún modelo. Último error: {last_err}")

tok, model, MODEL_NAME = load_seq2seq_model()

# Mantén el final del prompt si hay truncation (lo más importante suele estar al final)
tok.truncation_side = "left"
MAX_INPUT_LEN = 512
SCORING_BATCH_SIZE = 8  # baja a 4 si hay OOM

# -----------------------------
# 2) Etiquetas + definiciones (guía semántica)
# -----------------------------
LABELS = [e.lower() for e in emotions]
LABELS_STR = ", ".join(LABELS)

LABEL_DEFS = {
    "anger":   "angry, irritated, hostile, blaming, shouting, frustrated",
    "disgust": "disapproval, repulsion, contempt, 'gross', 'ew', 'that\'s awful'",
    "fear":    "scared, anxious, worried, panic, intimidated",
    "joy":     "happy, excited, pleased, amused, laughing",
    "neutral": "no strong emotion; matter-of-fact; informational; calm",
    "sadness": "sad, disappointed, down, regretful, grieving",
    "surprise":"surprised, shocked, amazed, unexpected reaction",
}
DEFS_BLOCK = "\n".join([f"- {k}: {LABEL_DEFS.get(k,'')}" for k in LABELS])

# -----------------------------
# 3) Few-shot balanceado (elige ejemplos cortos por clase)
# -----------------------------
SHOTS_PER_LABEL = 2      # sube a 3 si tu prompt no se trunca
MAX_EX_TOKENS = 120      # evita ejemplos demasiado largos

def _tok_len(s: str) -> int:
    return len(tok(s, add_special_tokens=False).input_ids)

def build_fewshot_block(df, shots_per_label=SHOTS_PER_LABEL, seed=SEED):
    rnd = random.Random(seed)
    blocks = []

    for lab in LABELS:
        subset = df[df["emotion"].str.lower() == lab].copy()
        if subset.empty:
            continue

        # Calculamos longitud de tokens y nos quedamos con los más cortos
        subset["n_tok"] = subset["input_text"].astype(str).apply(_tok_len)
        short = subset[subset["n_tok"] <= MAX_EX_TOKENS]
        if short.empty:
            short = subset.sort_values("n_tok").head(200)
        else:
            short = short.sort_values("n_tok").head(300)

        # Muestreo para diversidad dentro de los más cortos
        take_n = min(shots_per_label, len(short))
        picked = short.sample(n=take_n, random_state=rnd.randint(0, 10**9))

        for _, row in picked.iterrows():
            x = str(row["input_text"]).strip()
            blocks.append(
                "Example:\n"
                "```\n" + x + "\n```\n"
                f"Label: {lab}\n"
            )

    rnd.shuffle(blocks)
    return "\n".join(blocks)

FEWSHOT = build_fewshot_block(train_df)

# -----------------------------
# 4) Prompt ensembling + contextual calibration
# -----------------------------
def prompt_template_A(input_text: str) -> str:
    return (
        "Task: Classify the emotion expressed in the TARGET turn of a dialogue.\n"
        f"Choose exactly ONE label from: {LABELS_STR}\n"
        "Output ONLY the label (lowercase).\n\n"
        "Label definitions:\n"
        f"{DEFS_BLOCK}\n\n"
        f"{FEWSHOT}\n"
        "Now classify this target turn:\n"
        f"```\n{input_text.strip()}\n```\n"
        "Label:"
    )

def prompt_template_B(input_text: str) -> str:
    return (
        "You are an expert dialogue emotion classifier.\n"
        "Focus on the EMOTION of the target utterance, not the whole conversation.\n"
        f"Valid labels: {LABELS_STR}\n"
        "If there is no clear emotion, choose neutral.\n\n"
        f"{FEWSHOT}\n"
        "Dialogue turn:\n"
        f"{input_text.strip()}\n"
        "Answer with one label:\n"
        "Label:"
    )

def prompt_template_C(input_text: str) -> str:
    return (
        "Classify the emotion of the speaker in the TARGET turn.\n"
        f"Return one of: {LABELS_STR}\n\n"
        "Hints:\n"
        "- anger: irritation/blame\n"
        "- disgust: contempt/repulsion\n"
        "- fear: anxious/scared\n"
        "- joy: positive/excited\n"
        "- sadness: negative/down\n"
        "- surprise: shocked/unexpected\n"
        "- neutral: informational/calm\n\n"
        f"{FEWSHOT}\n"
        "TARGET:\n"
        f"{input_text.strip()}\n"
        "Label:"
    )

PROMPT_BUILDERS = [prompt_template_A, prompt_template_B, prompt_template_C]

# Prompt de calibración (content-free) para corregir prior de etiquetas
CALIBRATION_INPUT = (
    "Dialogue context (previous turns):\nNone\n\n"
    "Target turn to classify (emotion of THIS utterance):\n"
    "SPEAKER_0: [EMPTY]"
)

# -----------------------------
# 5) Label scoring batched (rápido)
# -----------------------------
pad_id = tok.pad_token_id
_label_ids = tok(LABELS, return_tensors="pt", padding=True, truncation=True).input_ids.to(device)
_decoder_input_ids = model.prepare_decoder_input_ids_from_labels(_label_ids)
L = _label_ids.size(0)

@torch.inference_mode()
def score_labels_batch(prompts, batch_size=8):
    """Devuelve una matriz (len(prompts), num_labels) con loss media por etiqueta."""
    all_losses = []

    for start in range(0, len(prompts), batch_size):
        chunk = prompts[start:start+batch_size]
        enc = tok(
            chunk,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LEN,
        ).to(device)

        P = enc.input_ids.size(0)

        # Repetimos cada prompt L veces para puntuar todas las etiquetas en 1 forward
        input_ids = enc.input_ids.repeat_interleave(L, dim=0)
        attention_mask = enc.attention_mask.repeat_interleave(L, dim=0)

        decoder_input_ids = _decoder_input_ids.repeat(P, 1)
        label_ids = _label_ids.repeat(P, 1)

        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
        )
        logits = out.logits  # (P*L, T, V)
        V = logits.size(-1)

        loss_tok = F.cross_entropy(
            logits.view(-1, V),
            label_ids.view(-1),
            ignore_index=pad_id,
            reduction="none",
        ).view(P*L, label_ids.size(1))

        denom = (label_ids != pad_id).sum(dim=1).clamp(min=1)
        loss_per = loss_tok.sum(dim=1) / denom   # (P*L,)
        loss_per = loss_per.view(P, L)           # (P, L)

        all_losses.append(loss_per.detach().cpu())

    return torch.cat(all_losses, dim=0)  # (N, L)

def get_calibration_losses():
    calib = []
    for builder in PROMPT_BUILDERS:
        p = builder(CALIBRATION_INPUT)
        losses = score_labels_batch([p], batch_size=1)[0]  # (L,)
        calib.append(losses)
    return calib

CALIB_LOSSES = get_calibration_losses()

@torch.inference_mode()
def predict_batch(texts, batch_size=8, show_progress=True):
    """Predice etiquetas para una lista de input_text."""
    preds = []
    it = range(0, len(texts), batch_size)
    if show_progress:
        it = tqdm(it, desc="Predict")

    for start in it:
        chunk_texts = texts[start:start+batch_size]

        # Ensemble: acumulamos losses de varias plantillas
        ensemble_losses = None

        for t_idx, builder in enumerate(PROMPT_BUILDERS):
            prompts = [builder(x) for x in chunk_texts]
            losses = score_labels_batch(prompts, batch_size=SCORING_BATCH_SIZE)  # (P, L)

            # Contextual calibration: restamos prior (por plantilla)
            losses = losses - CALIB_LOSSES[t_idx].unsqueeze(0)

            ensemble_losses = losses if ensemble_losses is None else (ensemble_losses + losses)

        ensemble_losses = ensemble_losses / len(PROMPT_BUILDERS)  # promedio

        best = torch.argmin(ensemble_losses, dim=1).tolist()
        preds.extend([LABELS[i] for i in best])

    return preds

# (Opcional) inspección rápida de un prompt
print("Ejemplo de prompt (template A)\n---\n", prompt_template_A(train_df.loc[0, "input_text"])[:900], "\n---")


Device: cuda


Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.82s/it]


✅ Cargado modelo: google/flan-t5-xl (quantized=True)
Ejemplo de prompt (template A)
---
 Task: Classify the emotion expressed in the TARGET turn of a dialogue.
Choose exactly ONE label from: anger, disgust, fear, joy, neutral, sadness, surprise
Output ONLY the label (lowercase).

Label definitions:
- anger: angry, irritated, hostile, blaming, shouting, frustrated
- disgust: disapproval, repulsion, contempt, 'gross', 'ew', 'that's awful'
- fear: scared, anxious, worried, panic, intimidated
- joy: happy, excited, pleased, amused, laughing
- neutral: no strong emotion; matter-of-fact; informational; calm
- sadness: sad, disappointed, down, regretful, grieving
- surprise: surprised, shocked, amazed, unexpected reaction

Example:
```
Dialogue context (previous turns):
SPEAKER_0: You're sure? You're absolutely sure? SPEAKER_2: Well, no. But, um, I bet she probably does. SPEAKER_0: Probably?

Target turn to classify (emotion of THIS utterance):
SPEAKER_0: Yeah, I don't like that wo 
---


In [16]:
from sklearn.metrics import classification_report

# Evaluación en el conjunto de test (muestra)
sample_size = 2000
test_sample = test_df.sample(n=min(sample_size, len(test_df)), random_state=42)

texts = test_sample["input_text"].tolist()
y_true = test_sample["emotion"].str.lower().tolist()
y_pred = predict_batch(texts)

print(
    classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        target_names=LABELS,
        zero_division=0,
    )
)

Predict: 100%|██████████| 250/250 [28:58<00:00,  6.95s/it]

              precision    recall  f1-score   support

       anger       0.41      0.40      0.41       270
     disgust       1.00      0.13      0.23        47
        fear       0.08      0.60      0.14        35
         joy       0.28      0.76      0.41       301
     neutral       0.82      0.14      0.24       982
     sadness       0.43      0.37      0.40       157
    surprise       0.36      0.59      0.44       208

    accuracy                           0.34      2000
   macro avg       0.48      0.43      0.32      2000
weighted avg       0.59      0.34      0.32      2000



## Notas / cómo exprimir más precisión (sin entrenar)

- Si tu GPU aguanta, el salto más grande suele venir de **`flan-t5-large` o `flan-t5-xl`**.
- Ajustes que normalmente mejoran *macro-F1*:
  - `SHOTS_PER_LABEL`: prueba 2 → 3 (si no hay truncation).
  - `MAX_EX_TOKENS`: baja si el prompt se vuelve muy largo.
  - `MAX_INPUT_LEN`: 512 suele ser el mejor compromiso (más puede aumentar coste/oom).
  - `PROMPT_BUILDERS`: puedes añadir otra plantilla y dejar el ensemble.
- Si ves **CUDA OOM**, baja `batch_size` en `predict_batch(...)` (por ejemplo 8 → 4).
